# Laboratorium 6: Lokalne modele językowe

W tym laboratorium porównujemy lokalne modele językowe jako komponenty systemu NLP. Obok jakości różnych wariantów odpowiedzi modelu interesuje nas również koszt uruchomienia, pamięć, opóźnienie, szybkość generowania i niezawodność wyjścia strukturalnego.

Główne pytanie laboratorium:

**Jak wybrać lokalny LLM, kiedy jednocześnie liczą się jakość, pamięć, opóźnienie i przewidywalność działania?**

Porównujemy co najmniej dwa lokalne modele lub dwie wyraźnie różne konfiguracje modeli.


## Po tych zajęciach student potrafi

- opisać środowisko uruchomieniowe lokalnego LLM;
- zebrać podstawowe informacje o modelach i kwantyzacji;
- oszacować koszt pamięci wag;
- zmierzyć cold start, TTFT i szybkość generowania;
- przygotować powtarzalny benchmark jakości;
- ocenić wyjście strukturalne;
- porównać dwie kwantyzacje tego samego modelu;
- porównać LLM z klasycznym baseline'em;
- analizować błędy modeli generatywnych;
- uzasadnić wybór modelu dla konkretnego zastosowania.


## Protokół

Wszystkie modele porównujemy na tym samym sprzęcie, na tych samych danych i przy możliwie zgodnych parametrach generowania. Jeżeli z powodów sprzętowych nie można uruchomić wybranego modelu, należy opisać ograniczenie i wybrać mniejszą konfigurację.

Nie zmieniamy promptu osobno dla każdego modelu tylko po to, aby poprawić jego wynik. Prompt stanowi część wspólnego benchmarku.

Parametry generowania są wspólne i zamrożone w `GEN_OPTIONS`: `temperature=0`, stały `seed`, stałe `num_ctx`. Limit `num_predict` ustalamy osobno dla każdego zadania, ale identycznie dla wszystkich modeli. Dla modeli z trybem „thinking" wyłączamy go (`think=False`) albo raportujemy go jako osobną konfigurację.

Wszystkie surowe odpowiedzi zapisujemy do `../artifacts` (JSONL lub parquet: tu potrzebny `pyarrow`). Analiza w zadaniach 7-9, a także w zadaniu dla chętnych, korzysta z zapisanych plików, a nie z ponownego generowania.

In [1]:

from pathlib import Path
import json, platform, time
import jsonschema
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests, psutil

from sklearn.metrics import accuracy_score, f1_score

RANDOM_STATE = 42
OLLAMA_BASE = "http://localhost:11434"
ARTIFACTS = Path("../artifacts")
ARTIFACTS.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 80)


In [2]:
GEN_OPTIONS = {"temperature": 0, "seed": RANDOM_STATE, "num_ctx": 4096}

def ollama_get(path):
    r = requests.get(f"{OLLAMA_BASE}{path}", timeout=10)
    r.raise_for_status()
    return r.json()

# /api/version -> wersja, /api/tags -> modele na dysku, /api/ps -> modele w pamięci (size, size_vram)
print("Ollama:", ollama_get("/api/version")["version"])

def unload(model):
    """Usuwa model z pamięci; kolejne wywołanie będzie cold startem."""
    r = requests.post(f"{OLLAMA_BASE}/api/generate",
                      json={"model": model, "keep_alive": 0}, timeout=120)
    r.raise_for_status()

def chat(model, prompt, *, system=None, fmt=None, num_predict=128, think=None):
    """Jedno wywołanie /api/chat ze streamingiem. Zwraca tekst i metryki."""
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": prompt})
    payload = {"model": model, "messages": messages, "stream": True,
               "options": {**GEN_OPTIONS, "num_predict": num_predict}}
    if fmt is not None:
        payload["format"] = fmt          # "json" albo słownik ze schematem JSON
    if think is not None:
        payload["think"] = think

    t0, ttft, chunks, last = time.perf_counter(), None, [], {}
    with requests.post(f"{OLLAMA_BASE}/api/chat", json=payload,
                       stream=True, timeout=600) as r:
        r.raise_for_status()
        for line in r.iter_lines():
            if not line:
                continue
            part = json.loads(line)
            if part.get("error"):
                raise RuntimeError(f"Ollama /api/chat: {part['error']}")
            piece = (part.get("message") or {}).get("content") or ""
            if piece and ttft is None:
                ttft = time.perf_counter() - t0
            chunks.append(piece)
            if part.get("done"):
                last = part
    if not last:
        raise RuntimeError(f"Niekompletna odpowiedź /api/chat dla {model}.")
    eval_s = last.get("eval_duration", 0) / 1e9
    return {
        "model": model,
        "text": "".join(chunks),
        "ttft_s": ttft,                                   # zegar klienta
        "total_s": time.perf_counter() - t0,              # zegar klienta
        "load_s": last.get("load_duration", 0) / 1e9,     # raport serwera
        "prompt_tokens": last.get("prompt_eval_count"),
        "output_tokens": last.get("eval_count"),
        "tokens_per_s": last.get("eval_count", 0) / eval_s if eval_s else None,
        "done_reason": last.get("done_reason"),           # "length" = ucięte przez num_predict
    }

Ollama: 0.40.14


# Zadanie 1: Audyt środowiska i modeli

Proszę przygotować opis środowiska zawierający:

- system operacyjny;
- CPU;
- RAM;
- GPU i VRAM, jeśli są dostępne;
- wersję Ollamy;
- listę lokalnie dostępnych modeli.

Następnie proszę wybrać co najmniej dwa modele lub konfiguracje do dalszego porównania i zapisać dla nich nazwę, rodzinę, przybliżoną liczbę parametrów, kwantyzację oraz rozmiar pliku.

Wynik proszę zapisać również do `artifacts/local_llm_environment.json`.


Zalecana konfiguracja (proszę pobrać przed zajęciami):

| rola | tag | przybliżony rozmiar |
|---|---|---:|
| mały model | `gemma3:1b` | 0.8 GB |
| większy model | `gemma3:4b` | 3.3 GB |
| druga kwantyzacja (zadanie 6) | `gemma3:4b-it-q8_0` | ok. 5 GB |

Przy 8 GB RAM i braku GPU zadanie 6 można wykonać na parze `gemma3:1b` (Q4_K_M) i `gemma3:1b-it-q8_0`.

Opcjonalnie, jako model z innej rodziny: `qwen3:4b`. Ten model ma tryb „thinking", dlatego w benchmarku uruchamiamy go z `think=False`. Wariant z `think=True` można dodać jako osobną konfigurację i porównać TTFT.

Inne modele są dopuszczalne, jeśli spełniają warunek zadania: co najmniej dwa rozmiary albo dwie kwantyzacje tego samego modelu.

In [3]:
# Proszę zaimplementować audyt środowiska i inwentaryzację modeli.

import subprocess

def gpu_info():
    if platform.system() == "Darwin" and platform.machine() == "arm64":
        return {"gpu": "Apple Silicon (GPU zintegrowane)", "vram_gb": None,
                "vram_note": "pamięć unifikowana, dzielona z RAM (brak osobnego limitu VRAM)"}
    return {"gpu": None, "vram_gb": None, "vram_note": "nie wykryto GPU / brak obsługi na tej platformie"}


def cpu_name():
    if platform.system() == "Darwin":
        try:
            out = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"],
                                 capture_output=True, text=True, timeout=5)
            if out.returncode == 0 and out.stdout.strip():
                return out.stdout.strip()
        except Exception:
            pass
    return platform.processor() or platform.machine()


def environment_info():
    vm = psutil.virtual_memory()
    return {"os": f"{platform.system()} {platform.release()} ({platform.version()})",
            "cpu": cpu_name(),
            "cpu_cores_logical": psutil.cpu_count(logical=True),
            "cpu_cores_physical": psutil.cpu_count(logical=False),
            "ram_gb": round(vm.total / 2**30, 1),
            **gpu_info(),
            "ollama_version": ollama_get("/api/version")["version"]}


def local_models():
    return [{"name": m["name"], "size_gb": round(m["size"] / 2**30, 2),
             "modified": m["modified_at"]} for m in ollama_get("/api/tags")["models"]]


SELECTED_MODELS = ["gemma3:1b", "gemma3:4b"]
FAMILY = {"gemma3": "Gemma 3 (Google DeepMind)"}


def ollama_show(tag):
    r = requests.post(f"{OLLAMA_BASE}/api/show", json={"model": tag}, timeout=30)
    r.raise_for_status()
    return r.json()


def model_details(tag, tags_info):
    size_bytes = next(m["size"] for m in tags_info if m["name"] == tag)
    family_key = tag.split(":")[0]
    show = ollama_show(tag)
    model_info = show.get("model_info", {})
    n_params = next((v for k, v in model_info.items() if k.endswith("parameter_count")), None)
    return {"name": tag, "family": FAMILY.get(family_key, family_key),
            "parameters_billion": round(n_params / 1e9, 2) if n_params else None,
            "quantization": show.get("details", {}).get("quantization_level"),
            "file_size_gb": round(size_bytes / 2**30, 2)}


env = environment_info()
tags_info = ollama_get("/api/tags")["models"]
models = local_models()
selected = [model_details(tag, tags_info) for tag in SELECTED_MODELS]

print("Środowisko:")
for k, v in env.items():
    print(f"  {k}: {v}")

print("\nLokalnie dostępne modele:")
display(pd.DataFrame(models))

print("\nModele wybrane do porównania:")
selected_df = pd.DataFrame(selected)
display(selected_df)

audit = {"environment": env, "local_models": models, "selected_for_comparison": selected}
with open(ARTIFACTS / "local_llm_environment.json", "w", encoding="utf-8") as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)
print(f"\nZapisano: {ARTIFACTS / 'local_llm_environment.json'}")


Środowisko:
  os: Darwin 25.6.0 (Darwin Kernel Version 25.6.0: Tue Aug 18 17:48:46 PDT 2026; root:xnu-12377.161.15.700.19~2/RELEASE_ARM64_T6041)
  cpu: Apple M4 Max
  cpu_cores_logical: 16
  cpu_cores_physical: 16
  ram_gb: 64.0
  gpu: Apple Silicon (GPU zintegrowane)
  vram_gb: None
  vram_note: pamięć unifikowana, dzielona z RAM (brak osobnego limitu VRAM)
  ollama_version: 0.40.14

Lokalnie dostępne modele:


,name,size_gb,modified
0,gemma3:4b,3.12,2026-11-17T10:22:24.933942589+01:00
1,gemma3:1b,0.75,2026-11-17T10:16:51.238355891+01:00
2,llama3.2:3b,1.88,2026-11-17T10:08:35.280682111+01:00
3,hf.co/speakleash/Bielik-4.5B-v3.0-Instruct-GGUF:Q8_0,4.71,2026-11-05T22:44:05.745055845+01:00



Modele wybrane do porównania:


,name,family,parameters_billion,quantization,file_size_gb
0,gemma3:1b,Gemma 3 (Google DeepMind),1.00,Q4_K_M,0.75
1,gemma3:4b,Gemma 3 (Google DeepMind),3.88,Q4_K_M,3.12



Zapisano: ../artifacts/local_llm_environment.json


# Zadanie 2: Pamięć i kwantyzacja

Proszę oszacować minimalny rozmiar samych wag dla modeli 1B, 4B i 8B przy zapisie FP16, INT8 i 4-bitowym.

Dla rzeczywiście uruchamianych modeli proszę porównać oszacowanie z rozmiarem pliku oraz, jeśli to możliwe, ze zmianą wykorzystania RAM/VRAM po załadowaniu modelu.

Proszę wyjaśnić, dlaczego rozmiar pliku, rozmiar samych wag i rzeczywiste zużycie pamięci nie muszą być identyczne.


Wskazówki:
- „4-bit" w plikach GGUF nie oznacza 4 bitów na każdą wagę. Proszę policzyć efektywną liczbę bitów na wagę jako `rozmiar_pliku * 8 / liczba_parametrów` i porównać ją z nominalną nazwą kwantyzacji.
- Ollama działa w osobnym procesie, więc pamięć procesu notebooka nic nie mówi o modelu. Proszę użyć `/api/ps` (pola `size` i `size_vram`) oraz różnicy `psutil.virtual_memory().used` przed załadowaniem i po nim.
- Proszę sprawdzić, jak zmienia się zużycie pamięci po zmianie `num_ctx` z 4096 na 8192 dla jednego modelu.

In [22]:
# Proszę zaimplementować oszacowanie pamięci i zebrać pomiary dla wybranych modeli.

BYTES_PER_PARAM = {"FP16": 2, "INT8": 1, "4-bit": 0.5}
SIZES_B = [1, 4, 8]

theoretical_df = pd.DataFrame(
    {fmt: [round(n * 1e9 * bpp / 2**30, 2) for n in SIZES_B] for fmt, bpp in BYTES_PER_PARAM.items()},
    index=[f"{n}B" for n in SIZES_B])
theoretical_df.index.name = "model"
print("Minimalny rozmiar samych wag [GB]:")
display(theoretical_df)


def memory_snapshot():
    ram_used_gb = psutil.virtual_memory().used / 2**30
    loaded = {m["name"]: m for m in ollama_get("/api/ps").get("models", [])}
    return ram_used_gb, loaded


def measure_model_memory(tag, num_ctx=4096):
    unload(tag)
    time.sleep(1)
    ram_before, _ = memory_snapshot()
    r = requests.post(f"{OLLAMA_BASE}/api/chat",
                      json={"model": tag, "messages": [{"role": "user", "content": "Cześć"}],
                            "stream": False, "options": {**GEN_OPTIONS, "num_ctx": num_ctx, "num_predict": 1}},
                      timeout=120)
    r.raise_for_status()
    ram_after, loaded = memory_snapshot()
    info = loaded.get(tag, {})
    return {"model": tag, "num_ctx": num_ctx,
            "ram_delta_gb": round(ram_after - ram_before, 2),
            "ollama_size_gb": round(info.get("size", 0) / 2**30, 2),
            "ollama_size_vram_gb": round(info.get("size_vram", 0) / 2**30, 2)}


memory_rows = [measure_model_memory(tag) for tag in SELECTED_MODELS]
memory_df = pd.DataFrame(memory_rows)
print("\nZużycie pamięci po załadowaniu (num_ctx=4096):")
display(memory_df)

comparison_rows = []
for tag in SELECTED_MODELS:
    details = next(d for d in selected if d["name"] == tag)
    file_size_bytes = details["file_size_gb"] * 2**30
    n_params = details["parameters_billion"] * 1e9
    comparison_rows.append(
        {
            "model": tag,
            "quantization": details["quantization"],
            "parameters_billion": details["parameters_billion"],
            "file_size_gb": details["file_size_gb"],
            "effective_bits_per_weight": round(file_size_bytes * 8 / n_params, 2)
        }
    )
comparison_df = pd.DataFrame(comparison_rows)
print("\nOszacowanie vs rzeczywistość:")
display(comparison_df)

ctx_rows = [measure_model_memory(SELECTED_MODELS[0], num_ctx=c) for c in (4096, 8192)]
ctx_df = pd.DataFrame(ctx_rows)
print(f"\nWpływ num_ctx na pamięć ({SELECTED_MODELS[0]}):")
display(ctx_df)


Minimalny rozmiar samych wag [GB]:


,FP16,INT8,4-bit
model,,,
1B,1.86,0.93,0.47
4B,7.45,3.73,1.86
8B,14.90,7.45,3.73



Zużycie pamięci po załadowaniu (num_ctx=4096):


,model,num_ctx,ram_delta_gb,ollama_size_gb,ollama_size_vram_gb
0,gemma3:1b,4096,0.31,0.81,0.81
1,gemma3:4b,4096,1.91,2.09,2.09



Oszacowanie vs rzeczywistość:


,model,quantization,parameters_billion,file_size_gb,effective_bits_per_weight
0,gemma3:1b,Q4_K_M,1.00,0.75,6.44
1,gemma3:4b,Q4_K_M,3.88,3.12,6.91



Wpływ num_ctx na pamięć (gemma3:1b):


,model,num_ctx,ram_delta_gb,ollama_size_gb,ollama_size_vram_gb
0,gemma3:1b,4096,0.30,0.81,0.81
1,gemma3:1b,8192,0.39,0.88,0.88


### Komentarz do wyników


- **Nominalna kwantyzacja a efektywna liczba bitów.** Oba modele są oznaczone jako `Q4_K_M`, czyli nazwa sugeruje 4 bity na wagę. Efektywnie wychodzi jednak 6.44 (gemma3:1b) i 6.91 (gemma3:4b) bita na wagę - więcej niż 4-bit z naszej tabeli (0.47-1.86 GB), a bliżej INT8. K-quanty w GGUF nie kwantyzują każdej wagi jednakowo: trzymają osobne skale/zero-pointy na blok wag (dodatkowy narzut) i część tensorów (np. embeddingi, warstwę wyjściową) w wyższej precyzji niż reszta sieci. Plik zawiera też metadane i tokenizator, które nie są wagami, ale są drobne w porównaniu z tym narzutem kwantyzacji.
- **Rozmiar pliku a `/api/ps` (`size`, `size_vram`).** Dla gemma3:1b plik to 0.75 GB, a Ollama po załadowaniu raportuje 0.81–0.88 GB - różnica to bufor KV cache i bufory obliczeniowe (aktywacje), których rozmiar zależy od `num_ctx`, a nie tylko od wag.
- **`size` i `size_vram` są identyczne.** Na Apple Silicon GPU i CPU dzielą tę samą pamięć unifikowaną, więc Ollama nie rozróżnia osobnej puli VRAM - cały model jest „w VRAM” w tym samym sensie, w jakim jest w RAM.
- **Delta `psutil.virtual_memory().used` jest szumiąca.** Dla gemma3:1b przy `num_ctx=4096` wyszło raz 1.13 GB, a przy drugim pomiarze (w teście `num_ctx`) tylko 0.34 GB — różnica wynika z innych procesów systemowych działających w tle, a nie z samego modelu. `/api/ps` jest w tym kontekście wiarygodniejszym źródłem niż `psutil`, bo mierzy pamięć konkretnie przypisaną do procesu Ollamy.

**Wpływ `num_ctx`.** Dla gemma3:1b zmiana z 4096 na 8192 zwiększyła zgłaszany rozmiar z 0.81 do 0.88 GB (ok. +70 MB) 0 to koszt większego bufora KV cache, niezależny od samych wag modelu. Przy dłuższym kontekście ten narzut rośnie liniowo z `num_ctx`, podczas gdy rozmiar samych wag się nie zmienia.

# Zadanie 3: Cold start, TTFT i szybkość generowania

Proszę użyć funkcji `chat()` i jednego krótkiego promptu z `num_predict=128`. Dla każdego modelu mierzymy dwa stany:

**Cold start** (3 powtórzenia, przed każdym `unload(model)`):
- `load_s` -- czas ładowania raportowany przez serwer;
- `ttft_s` -- czas do pierwszego fragmentu treści odpowiedzi, łącznie z ładowaniem.

**Stan ciepły** (jedno wywołanie rozgrzewające, potem 5 powtórzeń):
- `ttft_s`;
- `total_s`;
- `output_tokens`;
- `tokens_per_s`.

Proszę raportować medianę oraz min–max. Proszę skomentować, czy drugi i trzeci cold start różnią się od pierwszego i dlaczego (cache plików systemu operacyjnego).

In [4]:
# Proszę zaimplementować wspólny benchmark wydajności lokalnych modeli.

PROMPT = "What is the capital of France?"

rows = []
cold_runs = {}

for model in SELECTED_MODELS:
    unload(model)

    cold_load_s = []
    cold_ttft_s = []
    for _ in range(3):
        output = chat(model=model, prompt=PROMPT, num_predict=128)
        unload(model)
        cold_load_s.append(output["load_s"])
        cold_ttft_s.append(output["ttft_s"])
    cold_runs[model] = cold_load_s

    warm_ttft_s = []
    warm_total_s = []
    warm_output_tokens = []
    warm_tokens_per_s = []
    chat(model=model, prompt=PROMPT, num_predict=128)

    for _ in range(5):
        output = chat(model=model, prompt=PROMPT, num_predict=128)
        warm_ttft_s.append(output['ttft_s'])
        warm_total_s.append(output['total_s'])
        warm_output_tokens.append(output['output_tokens'])
        warm_tokens_per_s.append(output["tokens_per_s"])

    rows.append({
        "model": model,
        "cold_load_s_median": np.median(cold_load_s),
        "cold_load_s_min": np.min(cold_load_s),
        "cold_load_s_max": np.max(cold_load_s),
        "cold_ttft_s_median": np.median(cold_ttft_s),
        "cold_ttft_s_min": np.min(cold_ttft_s),
        "cold_ttft_s_max": np.max(cold_ttft_s),
        "warm_ttft_s_median": np.median(warm_ttft_s),
        "warm_ttft_s_min": np.min(warm_ttft_s),
        "warm_ttft_s_max": np.max(warm_ttft_s),
        "warm_total_s_median": np.median(warm_total_s),
        "warm_total_s_min": np.min(warm_total_s),
        "warm_total_s_max": np.max(warm_total_s),
        "warm_output_tokens_median": np.median(warm_output_tokens),
        "warm_output_tokens_min": np.min(warm_output_tokens),
        "warm_output_tokens_max": np.max(warm_output_tokens),
        "warm_tokens_per_s_median": np.median(warm_tokens_per_s),
        "warm_tokens_per_s_min": np.min(warm_tokens_per_s),
        "warm_tokens_per_s_max": np.max(warm_tokens_per_s),
    })

display(pd.DataFrame(rows).set_index("model").T)

print("\nCold start po kolei (1., 2., 3. powtórzenie), load_s:")
for model, vals in cold_runs.items():
    print(f"  {model}: {[round(v, 3) for v in vals]}")


model,gemma3:1b,gemma3:4b
cold_load_s_median,0.543493,1.121120
cold_load_s_min,0.529769,1.097226
cold_load_s_max,1.299263,2.330802
cold_ttft_s_median,0.574163,1.192120
cold_ttft_s_min,0.566182,1.179880
cold_ttft_s_max,1.360225,2.382283
warm_ttft_s_median,0.032809,0.065224
warm_ttft_s_min,0.022871,0.034345
warm_ttft_s_max,0.042216,0.077875
warm_total_s_median,0.477718,0.990436



Cold start po kolei (1., 2., 3. powtórzenie), load_s:
  gemma3:1b: [1.299, 0.543, 0.53]
  gemma3:4b: [2.331, 1.097, 1.121]


### Komentarz do wyników

**Czy 2. i 3. cold start różnią się od 1.?** Tak, dla obu modeli. 1. powtórzenie jest ok. 2× wolniejsze niż 2. i 3., a te dwa są prawie takie same:

| model | `load_s` (1., 2., 3. powtórzenie) | 1. vs 2./3. |
|---|---|---|
| `gemma3:1b` | 1,299 / 0,543 / 0,530 s | ok. 2,4× wolniej |
| `gemma3:4b` | 2,331 / 1,097 / 1,121 s | ok. 2,1× wolniej |

**Dlaczego.** `unload(model)` zwalnia wagi z pamięci Ollamy, ale nie czyści pamięci podręcznej systemu operacyjnego (OS file cache). Przy 1. powtórzeniu plik modelu trzeba wczytać z dysku, a po tym odczycie system trzyma jego strony w pamięci podręcznej. Powtórzenia 2. i 3. są więc dalej cold startem dla Ollamy (model ładuje się do jej pamięci od nowa), ale nie dla dysku, i dlatego są szybsze. Dla pomiarów oznacza to, że tylko 1. powtórzenie pokazuje pełny koszt zimnego startu z dysku. Mediana z trzech powtórzeń go zaniża.

# Zadanie 4: Klasyfikacja zero-shot

Proszę przygotować deterministyczną, możliwie zbalansowaną próbkę PolEmo 2.0 i jedną instrukcję klasyfikacyjną wspólną dla wszystkich modeli.

Dla każdego modelu proszę policzyć:

- accuracy;
- macro-F1;
- F1 dla każdej klasy;
- odsetek odpowiedzi, których nie da się poprawnie zinterpretować jako etykiety;
- medianę i średnią czasu na dokument (model rozgrzany);
- 95% przedział ufności macro-F1 (`macro_f1_ci`).

Odpowiedzi interpretujemy wyłącznie funkcją `parse_label`. Odpowiedź `INVALID` liczy się jako błąd klasyfikacji. Predykcje i surowe odpowiedzi proszę zapisać do `artifacts/lab6_classification_<model>.parquet`.

In [5]:
DATA_PATH = ARTIFACTS / "polemo2_clean.parquet"
HF_BASE = "https://huggingface.co/datasets/allegro/klej-polemo2-in/resolve/main/"

if DATA_PATH.exists():
    data = pd.read_parquet(DATA_PATH)
else:
    for name in ("valid.csv", "dev.csv", "validation.csv"):
        try:
            data = pd.read_csv(HF_BASE + name)
            break
        except Exception:
            continue
    else:
        raise FileNotFoundError("Brak artefaktu z lab. 1 i nie udało się pobrać zbioru walidacyjnego.")
    data["split"] = "validation"

data = data.rename(columns={"sentence": "text"})

LABEL_MAP = {
    "__label__meta_minus_m": "negatywna",
    "__label__meta_plus_m": "pozytywna",
    "__label__meta_zero": "neutralna",
    "__label__meta_amb": "niejednoznaczna",
    "minus": "negatywna",
    "plus": "pozytywna",
    "zero": "neutralna",
    "amb": "niejednoznaczna",
    "negatywna": "negatywna",
    "pozytywna": "pozytywna",
    "neutralna": "neutralna",
    "niejednoznaczna": "niejednoznaczna",
}
LABELS = list(dict.fromkeys(LABEL_MAP.values()))
source_col = "label" if "label" in data.columns else "target"

data["label"] = (
    data[source_col]
    .astype(str)
    .str.strip()
    .map(LABEL_MAP)
)

if data["label"].isna().any():
    raise ValueError(
        "Nierozpoznane etykiety PolEmo. "
        "Sprawdź kodowanie i źródło danych."
    )

if "split" not in data.columns:
    raise ValueError(
        "Brak kolumny split. Wskaż jawnie zbiór walidacyjny."
    )

valid_df = data[
    data["split"].astype(str).str.lower().isin(
        ["validation", "valid", "val", "dev"]
    )
].dropna(subset=["text", "label"]).copy()

if valid_df.empty:
    raise ValueError(
        "Nie znaleziono danych walidacyjnych. "
        "Sprawdź wartości w kolumnie split."
    )
    
assert set(valid_df["label"]) <= set(LABELS), "Etykiety spoza LABELS - proszę poprawić LABEL_MAP."

N_PER_CLASS = 100   # 100 dokumentów; na samym CPU można zmniejszyć do 15 i to opisać

parts = [
    group.sample(n=min(N_PER_CLASS, len(group)), random_state=RANDOM_STATE)
    for _, group in valid_df.groupby("label", sort=True)
]
classification_sample = (
    pd.concat(parts, ignore_index=True)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)
classification_sample.to_parquet(ARTIFACTS / "lab6_classification_sample.parquet", index=False)
display(classification_sample["label"].value_counts())

label
niejednoznaczna    100
negatywna          100
pozytywna          100
neutralna          100
Name: count, dtype: int64

In [10]:
classification_sample.head(4)

,text,target,split,n_chars,n_words,n_words_split,n_lines,has_url,has_email,has_punct_burst,has_newline,is_very_short,uppercase_ratio,high_uppercase,text_raw,text_norm,random_split,dup_in_split,label
0,"w ostatnich miesiącach ciąży była m pod jej opieką , wizyty były szybkie - sprawdzenie czy szyjka się nie skraca i czy kanał rodny zamknięty , no i sprawdzenie tętna dziecka . ...",amb,validation,639,105,125,1,False,False,False,False,False,0.006061,False,"w ostatnich miesiącach ciąży była m pod jej opieką , wizyty były szybkie - sprawdzenie czy szyjka się nie skraca i czy kanał rodny zamknięty , no i sprawdzenie tętna dziecka . ...","w ostatnich miesiącach ciąży była m pod jej opieką , wizyty były szybkie - sprawdzenie czy szyjka się nie skraca i czy kanał rodny zamknięty , no i sprawdzenie tętna dziecka . ...",train,False,niejednoznaczna
1,"Zdecydowała m się zarezerwować ten hotel ze względu na możliwość skorzystania z hotelowego Spa . Z recepcji otrzymała m mail - a , że wyznaczono mi wejście na godzinę 19 w dniu...",amb,validation,645,97,111,1,False,False,False,False,False,0.017341,False,"Zdecydowała m się zarezerwować ten hotel ze względu na możliwość skorzystania z hotelowego Spa . Z recepcji otrzymała m mail - a , że wyznaczono mi wejście na godzinę 19 w dniu...","zdecydowała m się zarezerwować ten hotel ze względu na możliwość skorzystania z hotelowego spa . z recepcji otrzymała m mail - a , że wyznaczono mi wejście na godzinę 19 w dniu...",test,False,niejednoznaczna
2,"Ilość gwiazdek nieadekwatna do oferowanego standardu ! Hotel w stanie budowy , brak wind , ogrzewania i klimatyzacji w pokojach , brak serwisu pokojowego - nikt nie sprząta pok...",minus,validation,279,38,46,1,False,False,False,False,False,0.008850,False,"Ilość gwiazdek nieadekwatna do oferowanego standardu ! Hotel w stanie budowy , brak wind , ogrzewania i klimatyzacji w pokojach , brak serwisu pokojowego - nikt nie sprząta pok...","ilość gwiazdek nieadekwatna do oferowanego standardu ! hotel w stanie budowy , brak wind , ogrzewania i klimatyzacji w pokojach , brak serwisu pokojowego - nikt nie sprząta pok...",train,False,negatywna
3,"Będąc w listopadzie nad morzem to człowiek liczy głównie na atrakcje hotelowe , gdyż pogoda nie zabardzo sprzyja spacerom po plaży . Wybralismy wraz ze znajomymi ten hotel poni...",amb,validation,926,148,157,1,False,False,False,False,False,0.013245,False,"Będąc w listopadzie nad morzem to człowiek liczy głównie na atrakcje hotelowe , gdyż pogoda nie zabardzo sprzyja spacerom po plaży . Wybralismy wraz ze znajomymi ten hotel poni...","będąc w listopadzie nad morzem to człowiek liczy głównie na atrakcje hotelowe , gdyż pogoda nie zabardzo sprzyja spacerom po plaży . wybralismy wraz ze znajomymi ten hotel poni...",test,False,niejednoznaczna


In [8]:
def parse_label(text):
    """Akceptuje samą nazwę klasy, ewentualnie w cudzysłowie lub z kropką."""
    value = str(text).strip().lower().strip(' \t\r\n"\'`.')
    return value if value in LABELS else "INVALID"

assert parse_label("Nie jest pozytywna.") == "INVALID"
assert parse_label('"Pozytywna."') == "pozytywna"

def macro_f1_ci(y_true, y_pred, n_boot=1000):
    rng = np.random.default_rng(RANDOM_STATE)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    scores = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y_true), len(y_true))
        scores.append(f1_score(y_true[i], y_pred[i], labels=LABELS,
                               average="macro", zero_division=0))
    return np.percentile(scores, [2.5, 97.5])

In [19]:
# Proszę zaimplementować wspólny benchmark klasyfikacji zero-shot.

PROMPT = ("Oceń wydźwięk opinii. Odpowiedz wyłącznie jedną etykietą, bez cudzysłowów, "
          "pogrubień i uzasadnienia. Dozwolone etykiety: " + ", ".join(LABELS) + ".")
NUM_PREDICT_CLS = 16

X = classification_sample["text"].tolist()
y = classification_sample["label"].tolist()


def classify_all(model):
    chat(model=model, prompt=X[0], system=PROMPT, num_predict=NUM_PREDICT_CLS)
    records = []
    for text, gold in zip(X, y):
        out = chat(model=model, prompt=text, system=PROMPT, num_predict=NUM_PREDICT_CLS)
        records.append({"text": text, "gold": gold, "raw_response": out["text"],
                        "pred": parse_label(out["text"]), "total_s": out["total_s"],
                        "prompt_tokens": out["prompt_tokens"], "output_tokens": out["output_tokens"]})
    return pd.DataFrame(records)


rows = []
classification_results = {}
for model in SELECTED_MODELS:
    df = classify_all(model)
    classification_results[model] = df
    df.to_parquet(ARTIFACTS / f"lab6_classification_{model.replace(':', '_').replace('/', '_')}.parquet", index=False)

    y_pred = df["pred"].tolist()
    lo, hi = macro_f1_ci(y, y_pred)
    per_class = f1_score(y, y_pred, labels=LABELS, average=None, zero_division=0)
    rows.append({
        "model": model,
        "accuracy": accuracy_score(y, y_pred),
        "macro-f1": f1_score(y, y_pred, labels=LABELS, average="macro", zero_division=0),
        **{f"f1_{label}": score for label, score in zip(LABELS, per_class)},
        "macro-f1 95% CI": f"[{lo:.3f}, {hi:.3f}]",
        "invalid_ratio": (df["pred"] == "INVALID").mean(),
        "time_median_ms": df["total_s"].median() * 1000,
        "time_mean_ms": df["total_s"].mean() * 1000,
    })

classification_df = pd.DataFrame(rows).set_index("model")
display(classification_df.T)


model,gemma3:1b,gemma3:4b
accuracy,0.475,0.5575
macro-f1,0.348705,0.455522
f1_negatywna,0.594595,0.729927
f1_pozytywna,0.677419,0.69403
f1_neutralna,0.122807,0.379085
f1_niejednoznaczna,0.0,0.019048
macro-f1 95% CI,"[0.317, 0.381]","[0.413, 0.492]"
invalid_ratio,0.0025,0.0
time_median_ms,42.162062,222.420875
time_mean_ms,48.359653,237.378928


# Zadanie 5: Wyjście strukturalne

Proszę wykorzystać przygotowany zestaw przypadków ekstrakcyjnych i porównać trzy tryby dla każdego modelu:

1. `free` -- sama instrukcja z opisem kluczy, bez parametru `format`;
2. `json` -- `format="json"`;
3. `schema` -- `format=IE_SCHEMA` (dekodowanie ograniczone schematem).

W trybie `schema` poprawność składni jest wymuszona przez dekoder, więc nie jest miarą jakości modelu. Proszę sprawdzić, czy wymuszenie formatu zmienia poprawność treści.

Dla każdego modelu proszę zmierzyć:

- poprawność składni JSON;
- zgodność ze schematem;
- poprawność pól;
- poprawność całego rekordu;
- średni czas odpowiedzi.

Proszę odróżnić błąd treści od błędu formatu.

In [12]:

IE_CASES = [
    {"text": "W czwartek po południu padał deszcz.",
     "gold": {"person": None, "organization": None, "location": None}},
    {"text": "Uniwersytet Warszawski ogłosił nowy program stypendialny.",
     "gold": {"person": None, "organization": "Uniwersytet Warszawski", "location": None}},
    {"text": "Marii Skłodowskiej-Curie poświęcono wystawę w Paryżu.",
     "gold": {"person": "Marii Skłodowskiej-Curie", "organization": None, "location": "Paryżu"}},
    {"text": "Do Gdańska przyjechał Adam Małysz na zaproszenie Polskiego Związku Narciarskiego.",
     "gold": {"person": "Adam Małysz", "organization": "Polskiego Związku Narciarskiego", "location": "Gdańska"}},
    {"text": "Allegro otworzyło biuro w Poznaniu, a kieruje nim Ewa Nowak.",
     "gold": {"person": "Ewa Nowak", "organization": "Allegro", "location": "Poznaniu"}},
    {"text": "Prezes Orlenu spotkał się z dziennikarzami w Płocku.",
     "gold": {"person": None, "organization": "Orlenu", "location": "Płocku"}},
]

IE_SCHEMA = {
    "type": "object",
    "properties": {
        "person": {"anyOf": [{"type": "string"}, {"type": "null"}]},
        "organization": {"anyOf": [{"type": "string"}, {"type": "null"}]},
        "location": {"anyOf": [{"type": "string"}, {"type": "null"}]},
    },
    "required": ["person", "organization", "location"],
    "additionalProperties": False,
}

IE_PROMPT = '''Wyodrębnij z polskiego tekstu jedną osobę, jedną organizację i jedną lokalizację.
Zwracaj literalne fragmenty tekstu, w tej samej formie gramatycznej co w tekście. Jeśli kategorii nie ma, zwróć null.
Odpowiedz wyłącznie obiektem JSON z kluczami 'person', 'organization', 'location'.

Tekst:
'''


In [13]:
def score_ie(raw, gold):
    out = {"json_ok": False, "schema_ok": False, "fields_ok": 0.0, "record_ok": False}
    try:
        obj = json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        return out                               # błąd formatu
    out["json_ok"] = True
    try:
        jsonschema.validate(obj, IE_SCHEMA)
        out["schema_ok"] = True
    except jsonschema.ValidationError:
        pass                            # błąd schematu
    if isinstance(obj, dict):
        out["fields_ok"] = sum((k in obj) and (obj[k] == v)
                               for k, v in gold.items()) / len(gold)
    out["record_ok"] = bool(out["schema_ok"] and out["fields_ok"] == 1.0)
    return out

assert score_ie("{}", {"person": None, "organization": None, "location": None})["fields_ok"] == 0.0

In [18]:
import re

# Proszę zaimplementować benchmark wyjścia strukturalnego.

MODES = {"free": None, "json": "json", "schema": IE_SCHEMA}

def strip_fences(raw):
    m = re.search(r"```(?:json)?\s*(.*?)\s*```", raw, flags=re.S)
    return m.group(1) if m else raw

ie_rows = []
for model in SELECTED_MODELS:
    chat(model=model, prompt=IE_PROMPT + IE_CASES[0]["text"], num_predict=128)   #
    for mode_name, fmt in MODES.items():
        strict, lenient, times = [], [], []
        for case in IE_CASES:
            out = chat(model=model, prompt=IE_PROMPT + case["text"], fmt=fmt, num_predict=128)
            strict.append(score_ie(out["text"], case["gold"]))
            lenient.append(score_ie(strip_fences(out["text"]), case["gold"]))
            times.append(out["total_s"])
        s, l = pd.DataFrame(strict), pd.DataFrame(lenient)
        ie_rows.append({"model": model,
                        "tryb": mode_name,
                        "json_ok": s["json_ok"].mean(),
                        "schema_ok": s["schema_ok"].mean(),
                        "fields_ok": s["fields_ok"].mean(),
                        "record_ok": s["record_ok"].mean(),
                        "fields_ok (bez ```)": l["fields_ok"].mean(),
                        "record_ok (bez ```)": l["record_ok"].mean(),
                        "time_mean_ms": np.mean(times) * 1000})
display(pd.DataFrame(ie_rows).set_index(["model", "tryb"]).round(3))

json_ok  schema_ok  fields_ok  record_ok  \
model     tryb                                               
gemma3:1b free        0.0        0.0      0.000        0.0   
          json        1.0        1.0      0.278        0.0   
          schema      1.0        1.0      0.278        0.0   
gemma3:4b free        0.0        0.0      0.000        0.0   
          json        1.0        1.0      0.722        0.5   
          schema      1.0        1.0      0.722        0.5   

                  fields_ok (bez ```)  record_ok (bez ```)  time_mean_ms  
model     tryb                                                            
gemma3:1b free                  0.556                0.167       218.494  
          json                  0.278                0.000       210.656  
          schema                0.278                0.000       216.842  
gemma3:4b free                  0.722                0.500       432.756  
          json                  0.722                0.500       430.569  
          schema                0.722                0.500       430.940

# Zadanie 6: Dwie kwantyzacje tego samego modelu

Jeżeli lokalne środowisko na to pozwala, proszę wybrać jeden model dostępny w dwóch kwantyzacjach i porównać je na niewielkiej wspólnej próbce.

Należy porównać:

- rozmiar modelu;
- pamięć;
- TTFT;
- tokeny/s;
- macro-F1 w klasyfikacji;
- poprawność JSON.

Proszę sprawdzić, czy zaobserwowana różnica macro-F1 jest istotna praktycznie i jak duża jest niepewność oszacowania. Ponieważ oba modele oceniamy na tych samych dokumentach, proszę przeprowadzić również analizę sparowaną. Proszę dodatkowo podać porównanie sparowane na tych samych dokumentach:

- odsetek dokumentów, na których obie kwantyzacje dały tę samą predykcję;
- liczbę zmian „poprawna → błędna" i „błędna → poprawna".

Dostępne kwantyzacje danego modelu proszę sprawdzić na liście tagów w bibliotece Ollamy (np. wariant `q4_K_M` i `q8_0`).

Jeżeli druga kwantyzacja nie może zostać uruchomiona na dostępnym sprzęcie, proszę udokumentować ograniczenie i wykonać porównanie na podstawie dwóch konfiguracji różniących się rozmiarem modelu.

In [23]:
# Proszę przeprowadzić kontrolowane porównanie dwóch kwantyzacji lub dwóch rozmiarów modelu.


QUANT_MODELS = ["gemma3:4b", "gemma3:4b-it-q8_0"]
Q4, Q8 = QUANT_MODELS
SPEED_PROMPT = "What is the capital of France?"

tags_info = ollama_get("/api/tags")["models"]
quant_info = {tag: model_details(tag, tags_info) for tag in QUANT_MODELS}

for tag in QUANT_MODELS:
    if tag not in classification_results:
        classification_results[tag] = classify_all(tag)
gold = classification_results[Q4]["gold"].tolist()
pred = {tag: classification_results[tag]["pred"].tolist() for tag in QUANT_MODELS}


def warm_speed(tag, n=5):
    chat(model=tag, prompt=SPEED_PROMPT, num_predict=128)
    runs = [chat(model=tag, prompt=SPEED_PROMPT, num_predict=128) for _ in range(n)]
    return {"ttft_s": np.median([r["ttft_s"] for r in runs]),
            "tokens_per_s": np.median([r["tokens_per_s"] for r in runs])}


def json_scores(tag):
    scores = [score_ie(chat(model=tag, prompt=IE_PROMPT + c["text"], fmt="json", num_predict=128)["text"], c["gold"])
              for c in IE_CASES]
    return pd.DataFrame(scores).mean()


def paired_macro_f1_diff_ci(y_true, pred_a, pred_b, n_boot=1000):
    rng = np.random.default_rng(RANDOM_STATE)
    y_true, pred_a, pred_b = map(np.asarray, (y_true, pred_a, pred_b))
    diffs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y_true), len(y_true))
        diffs.append(f1_score(y_true[i], pred_b[i], labels=LABELS, average="macro", zero_division=0)
                     - f1_score(y_true[i], pred_a[i], labels=LABELS, average="macro", zero_division=0))
    return np.percentile(diffs, [2.5, 97.5])


rows = []
for tag in QUANT_MODELS:
    mem = measure_model_memory(tag)
    lo, hi = macro_f1_ci(gold, pred[tag])
    ie = json_scores(tag)
    rows.append({"model": tag, "quantization": quant_info[tag]["quantization"],
                 "file_size_gb": quant_info[tag]["file_size_gb"], "loaded_gb": mem["ollama_size_gb"],
                 **warm_speed(tag),
                 "macro_f1": f1_score(gold, pred[tag], labels=LABELS, average="macro", zero_division=0),
                 "macro_f1_ci": f"[{lo:.3f}, {hi:.3f}]",
                 "json_ok": ie["json_ok"], "record_ok": ie["record_ok"]})
display(pd.DataFrame(rows).set_index("model").T)
diff_lo, diff_hi = paired_macro_f1_diff_ci(gold, pred[Q4], pred[Q8])
print(f"\nRóżnica macro-F1 ({Q8} minus {Q4}): 95% CI [{diff_lo:+.3f}, {diff_hi:+.3f}]")

model,gemma3:4b,gemma3:4b-it-q8_0
quantization,Q4_K_M,Q8_0
file_size_gb,3.12,4.65
loaded_gb,2.09,4.13
ttft_s,0.036911,0.044639
tokens_per_s,64.21628,57.620878
macro_f1,0.455522,0.493913
macro_f1_ci,"[0.413, 0.492]","[0.449, 0.539]"
json_ok,1.0,1.0
record_ok,0.5,0.5



Różnica macro-F1 (gemma3:4b-it-q8_0 minus gemma3:4b): 95% CI [+0.010, +0.070]


te same predykcje     0.9225
poprawna → błędna     7.0000
błędna → poprawna    15.0000
dtype: float64

In [27]:
ok_a = np.array(pred[Q4]) == np.array(gold)
ok_b = np.array(pred[Q8]) == np.array(gold)
display({"te same predykcje": (np.array(pred[Q4]) == np.array(pred[Q8])).mean(),
                   "poprawna → błędna": int((ok_a & ~ok_b).sum()),
                   "błędna → poprawna": int((~ok_a & ok_b).sum())})

{'te same predykcje': np.float64(0.9225),
 'poprawna → błędna': 7,
 'błędna → poprawna': 15}

### Komentarz do wyników

Porównanie `gemma3:4b` w kwantyzacjach Q4_K_M i Q8_0 (400 dokumentów, ta sama próbka i ta sama kolejność):

| | Q4_K_M | Q8_0 | zmiana |
|---|---:|---:|---:|
| plik na dysku | 3,12 GB | 4,65 GB | 1,5× |
| pamięć po załadowaniu | 2,09 GB | 4,13 GB | ok. 2× |
| TTFT (stan ciepły) | 37 ms | 45 ms | +8 ms |
| tokeny/s | 64,2 | 57,6 | −10% |
| macro-F1 | 0,456 | 0,494 | +0,038 |
| 95% CI macro-F1 | [0,413; 0,492] | [0,449; 0,539] | |
| poprawność JSON / całego rekordu | 1,0 / 0,5 | 1,0 / 0,5 | bez zmian |

**Czy różnica macro-F1 jest istotna?** Statystycznie tak: sparowany 95% przedział różnicy (Q8 minus Q4) wynosi [+0,010; +0,070] i nie zawiera zera. Przedziały obu modeli liczone osobno nachodzą na siebie, więc same z siebie nie pozwalałyby wyciągnąć takiego wniosku. Dopiero analiza sparowana uwzględnia, że oba modele oceniamy na tych samych dokumentach.

**Analiza sparowana.** Q4 i Q8 dały tę samą predykcję dla 92,25% dokumentów (369 z 400). Spośród 31 dokumentów, na których się różnią, poprawność zmieniła się w 22: 15 zmian „błędna → poprawna” i 7 „poprawna → błędna”. W pozostałych 9 oba modele się mylą, ale wybierają różne etykiety. Netto Q8 ma o 8 poprawnych dokumentów więcej (+2 punkty procentowe accuracy).

**Czy różnica jest istotna praktycznie?** Niewielka i niepewna. Punktowo to +0,038, ale dolna granica przedziału (+0,010) jest praktycznie bez znaczenia, a górna (+0,070) to niewiele przy tak niskim poziomie wyjściowym (macro-F1 ok. 0,46–0,49 dla czterech klas). Za to koszt jest wyraźny: ok. 2× więcej pamięci, 1,5× większy plik i o 10% wolniejsze generowanie. Na tym sprzęcie (64 GB pamięci) ten koszt jest do przyjęcia, więc Q8 jest rozsądnym wyborem. Przy ograniczonej pamięci (np. 8 GB) lepszy będzie Q4_K_M, który daje te same predykcje w ponad 92% przypadków.

**Poprawność JSON.** `json_ok` = 1,0 dla obu, ale w trybie `format="json"` składnię wymusza dekoder, więc nie jest to miara jakości modelu. Poprawność całego rekordu wynosi 0,5 dla obu (3 z 6 przypadków), czyli kwantyzacja nie zmieniła poprawności treści. Próbka ma jednak tylko 6 przypadków, więc nie wyklucza to różnicy.

**Ograniczenia.** Jedna próbka (400 dokumentów), jeden prompt i jedno ustawienie generowania (`temperature=0`, stały `seed`). Przedział ufności uwzględnia tylko losowość doboru dokumentów. TTFT i tokeny/s zmierzono na jednym krótkim zapytaniu.

# Zadanie 7: Porównanie z klasycznym baseline'em

Proszę zestawić najlepsze lokalne LLM-y z wynikiem `TF-IDF + Linear SVM` z laboratorium 2.

| system | macro-F1 | ms/dokument | pamięć / rozmiar | poprawność formatu |
|---|---:|---:|---:|---:|
| TF-IDF + Linear SVM | | | | nie dotyczy |
| LLM A | | | | |
| LLM B | | | | |

Baseline proszę ocenić na dokładnie tej samej próbce (`lab6_classification_sample.parquet`) i zmierzyć jego czas na tym samym sprzęcie. Jeżeli model z laboratorium 2 nie jest dostępny, proszę wytrenować go ponownie na zbiorze treningowym PolEmo 2.0.

Kolumnę „pamięć / rozmiar" proszę rozbić na dwie: rozmiar na dysku i pamięć po załadowaniu. Proszę zaznaczyć, że porównanie nie jest symetryczne: SVM widział dane treningowe, a LLM działa zero-shot.

In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

final_pipe = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_features=50_000, sublinear_tf=True)),
        ("logreg", LogisticRegression(C=5.5, max_iter=1000, random_state=42))
    ]
)
train_df = data[
    data["split"].astype(str).str.lower().isin(
        ["train"]
    )
]

X_train = train_df["text"]
y_train = train_df["label"]
final_pipe.fit(X_train, y_train)

X_base = classification_sample["text"].tolist()
y_base = classification_sample["label"].tolist()

t0 = time.perf_counter()
y_pred_base = final_pipe.predict(X_base).tolist()
ms_per_doc = (time.perf_counter() - t0) * 1000 / len(X_base)

lo, hi = macro_f1_ci(y_base, y_pred_base)
macro_f1_base = f1_score(y_base, y_pred_base, labels=LABELS, average="macro", zero_division=0)
print(f"Macro-F1: {macro_f1_base:.3f} [{lo:.3f}, {hi:.3f}], czas: {ms_per_doc:.3f} ms/dok")

Macro-F1: 0.793 [0.753, 0.830], czas: 0.064 ms/dok
0


| system | macro-F1 (95% CI) | ms/dokument | rozmiar na dysku | pamięć po załadowaniu | poprawność formatu |
|---|---:|---:|---:|---:|---:|
| TF-IDF + regresja logistyczna (C=5.5, trening na `train`) | 0,793 [0,753; 0,830] | 0,064 | 3,46 MB | ok. 52 MB | nie dotyczy |
| LLM A: `gemma3:4b-it-q8_0` (Q8_0) | 0,494 [0,449; 0,539] | 224 (mediana) | 4,65 GB | 4,13 GB | 1,000 |
| LLM B: `gemma3:4b` (Q4_K_M) | 0,456 [0,413; 0,492] | 222 (mediana) | 3,12 GB | 2,09 GB | 1,000 |

# Zadanie 8: Analiza 20 przypadków

Proszę przygotować dokładnie 20 przypadków pokazujących ograniczenia lokalnych LLM-ów.

Co najmniej 10 powinno pochodzić z klasyfikacji, a co najmniej 5 z zadania strukturalnego (z dowolnego trybu: `free`, `json`, `schema`). Przypadki proszę wybierać z zapisanych surowych odpowiedzi. Jeżeli błędów strukturalnych jest mniej niż 5, proszę to odnotować i uzupełnić tabelę przypadkami z klasyfikacji.

Tabela:

| nr | model | zadanie | wejście | oczekiwany wynik | wynik modelu | typ problemu | komentarz |
|---:|---|---|---|---|---|---|---|

Proszę stosować krótkie kategorie, np. `WRONG_LABEL`, `FORMAT`, `SCHEMA`, `HALLUCINATION`, `OVERLONG`, `INCONSISTENT`, `OTHER`.


| nr | model | zadanie | wejście | oczekiwany wynik | wynik modelu | typ problemu | komentarz |
|---:|---|---|---|---|---|---|---|
| 1 | gemma3:4b | klasyfikacja | [#182] „Jej głównymi udziałowcami są Emirates Investments Group , Majid Al Futtaim Group and Abraaj Cap…” | neutralna | pozytywna | `WRONG_LABEL` | Informacyjny tekst o firmie (udziałowcy, zasięg działania) oceniony jako pozytywny. Model 1b również wskazał pozytywną. |
| 2 | gemma3:4b | klasyfikacja | [#212] „Stanowisko managera obiektu noclegowego The Bonerowski Palace w Krakowie objęła 1 czerwca br .…” | neutralna | pozytywna | `WRONG_LABEL` | Wiadomość prasowa o objęciu stanowiska, bez oceny, uznana za pozytywną. Model 1b również. |
| 3 | gemma3:1b | klasyfikacja | [#364] „Jednocześnie sami goście nie są przywiązani żadnymi zobowiązaniami . A jeśli już to pozornymi –…” | neutralna | Negatywna | `WRONG_LABEL` | Esejowy tekst o branży hotelowej uznany za negatywny. Model 4b również wskazał negatywną. |
| 4 | gemma3:4b | klasyfikacja | [#268] „Pokoje są zbyt małe , raczej niewygodne i bardzo proste . Mieli śmy problem z parkowaniem - nie…” | niejednoznaczna | negatywna | `WRONG_LABEL` | Opinia mieszana (wady, ale „personel raczej pomocny” i „blisko do lotniska”). Model wybiera dominujący, negatywny ton i nie używa klasy niejednoznaczna. |
| 5 | gemma3:4b | klasyfikacja | [#331] „Hotel lekko zapyziały . Hol , bar i restauracja OK . Odnowione , ale pokoje proszą o zmianę wys…” | niejednoznaczna | negatywna | `WRONG_LABEL` | Opinia mieszana („Hol, bar i restauracja OK”, ale pokoje i łazienki „proszą o zmianę wystroju”) oceniona jako negatywna. |
| 6 | gemma3:4b | klasyfikacja | [#299] „Może jeszcze za wcześnie na taką ocenę , bo jak do tej pory była m u niego na pierwszej wizycie…” | niejednoznaczna | pozytywna | `WRONG_LABEL` | Autor zastrzega „może jeszcze za wcześnie na taką ocenę”. Model pomija zastrzeżenie i ocenia po słowach „kompetentny i sympatyczny”. |
| 7 | gemma3:1b / gemma3:4b | klasyfikacja | [#286] „Lokalizacja super przy dużym placu gdzie w sobotę wystawił się targ z winami , seriami itd poko…” | niejednoznaczna | 1b: Pozytywna / 4b: neutralna | `INCONSISTENT` | Modele rozstrzygają tę samą opinię mieszaną („Lokalizacja super”, ale „brak ubikacji w pokojach”) w różne strony i żaden nie trafia. |
| 8 | gemma3:1b | klasyfikacja | [#203] „Waham się pomiędzy dobry a neutralny . Panią doktor poznała m na wizycie w Luxmedzie , była doś…” | niejednoznaczna | Neutralny | `FORMAT` | Autor pisze „Waham się pomiędzy dobry a neutralny”. Model przepisał słowo z recenzji w formie męskiej, więc `parse_label` zwraca INVALID. To jedyna niepoprawna etykieta wśród 400 dokumentów. |
| 9 | gemma3:1b | klasyfikacja | [#130] „Pokojom nie można niczego zarzucić , analogiczny do tej samej klasy w Trójmieście . Jedzenie ba…” | pozytywna | negatywna | `WRONG_LABEL` | Pochwała zapisana z negacją („nie można niczego zarzucić”) pomylona z krytyką. Model 4b odpowiada poprawnie. |
| 10 | gemma3:4b | klasyfikacja | [#166] „Świetna atmosfera , doskonałe posiłki , wygodny pokój i bardzo miła obsługa . Uwaga na żony - d…” | pozytywna | negatywna | `WRONG_LABEL` | Żart „Uwaga na żony - duży wybór zabiegów SPA powoduje wzrost wydatków” odczytany jako krytyka, mimo samych pochwał. Model 1b odpowiada poprawnie. |
| 11 | gemma3:1b / gemma3:4b | klasyfikacja | [#91] „Hotel jak na 4 gwiazdki spełnia oczekiwania . Jedzenie w miarę choć szału nie było . Dobra baza…” | pozytywna | 1b: Negatywna / 4b: niejednoznaczna | `INCONSISTENT` | Umiarkowanie pozytywna opinia („spełnia oczekiwania”, ale „szału nie było”). Modele wybierają różne, błędne etykiety. |
| 12 | gemma3:1b | klasyfikacja | [#165] „Do Pani dermatolog trafiła m z nawrotem trądziku i LZS skóry głowy . Trądzik był umiarkowany ,…” | negatywna | Niejednoznaczna | `WRONG_LABEL` | Długi tekst (2227 znaków) z krytyczną opinią o leczeniu oceniony jako niejednoznaczny. Model 4b odpowiada poprawnie. |
| 13 | gemma3:4b | struktura (free) | „W czwartek po południu padał deszcz.” | {"person": null, "organization": null, "location": null} | ta sama treść, ale owinięta w blok ```json … ``` | `FORMAT` | Treść poprawna, lecz `json.loads` odrzuca znaczniki markdown. W trybie free tak odpowiadają oba modele w 6 z 6 przypadków (json_ok = 0,0). |
| 14 | gemma3:1b | struktura (free) | „Uniwersytet Warszawski ogłosił nowy program stypendialny.” | person: null | person: "nieznany" | `HALLUCINATION` | Model wpisuje „nieznany” zamiast null. Napis nie występuje w tekście. |
| 15 | gemma3:1b | struktura (json) | „Uniwersytet Warszawski ogłosił nowy program stypendialny.” | organization: "Uniwersytet Warszawski"; person: null; location: null | person: "Uniwersytet"; organization: "Warszawski"; location: "Warszawa" | `HALLUCINATION` | Nazwa rozbita na dwa pola, a lokalizacja „Warszawa” wymyślona (w zdaniu jest tylko przymiotnik „Warszawski”). |
| 16 | gemma3:1b | struktura (schema) | „W czwartek po południu padał deszcz.” | {"person": null, "organization": null, "location": null} | wszystkie trzy pola: napis "null" | `OTHER` | Zgodne ze schematem (pole może być napisem), ale model wpisał napis „null” zamiast wartości null. Walidacja schematu tego nie wykrywa. |
| 17 | gemma3:1b | struktura (schema) | „Do Gdańska przyjechał Adam Małysz na zaproszenie Polskiego Związku Narciarskiego.” | location: "Gdańska" | location: "Do Gdańska" | `INCONSISTENT` | Przyimek wchodzi do wartości, wbrew poleceniu o literalnym fragmencie nazwy. „Do Gdańska” nie jest nazwą lokalizacji. |
| 18 | gemma3:4b | struktura (json) | „Do Gdańska przyjechał Adam Małysz na zaproszenie Polskiego Związku Narciarskiego.” | organization: "Polskiego Związku Narciarskiego"; location: "Gdańska" | organization: "Polski Związek Narciarski"; location: "Gdańsk" | `INCONSISTENT` | Mimo polecenia o tej samej formie gramatycznej model sprowadza nazwy do mianownika. Takiej wartości nie da się odnaleźć w tekście. |
| 19 | gemma3:4b | struktura (schema) | „Prezes Orlenu spotkał się z dziennikarzami w Płocku.” | person: null; organization: "Orlenu" | person: "Prezes Orlenu"; organization: "Orlen" | `OTHER` | Stanowisko wzięte za osobę (gold: null). Odpowiedź zgodna ze schematem, ale błędna treściowo. |
| 20 | gemma3:4b | struktura (schema) | „Allegro otworzyło biuro w Poznaniu, a kieruje nim Ewa Nowak.” | location: "Poznaniu" | location: "Poznań" | `INCONSISTENT` | Wymuszenie schematu nie zmienia zachowania: format jest poprawny, ale forma zlematyzowana zamiast literalnej. |

# Zadanie 9: Interpretacja wyników

To zadanie nie wymaga nowej implementacji.

Proszę odpowiedzieć:

1. Czy większy model był zawsze lepszy?
2. Jaki koszt był najbardziej odczuwalny: pamięć, TTFT czy szybkość generowania?
3. Czy kwantyzacja zmieniła jakość na tyle, by miało to znaczenie praktyczne?
4. Czy odpowiedzi poprawne formalnie były też poprawne merytorycznie? Proszę podać liczbę rekordów z poprawnym JSON i błędną treścią oraz porównać tryby `free`, `json` i `schema`.
5. Jakie dwa ograniczenia benchmarku najbardziej utrudniają uogólnienie wyników?

### Odpowiedzi do zadania 9

**1. Czy większy model był zawsze lepszy?**

Nie zawsze. `gemma3:4b` jest lepszy w jakości klasyfikacji (macro-F1 0,456 wobec 0,349, accuracy 0,558 wobec 0,475) i w zadaniu strukturalnym (poprawne rekordy 3 z 6 wobec 0 z 6 w trybach `json` i `schema`). Ale:
- Na 400 dokumentach 4b ma 44 dokumenty poprawne tam, gdzie 1b się myli, a 1b ma 11 poprawnych tam, gdzie 4b się myli. Większy model poprawia wynik, ale go nie zastępuje.
- Przewaga dotyczy głównie klasy `neutralna` (29 wobec 7 poprawnych na 100). W klasie `niejednoznaczna` oba modele są praktycznie bezużyteczne (1b: 0/100, 4b: 1/100), więc większy model tego problemu nie rozwiązał.
- Pod względem kosztu mniejszy model jest po prostu lżejszy i szybszy, więc przy kryterium pamięci lub czasu wygrywa `gemma3:1b`. Nie „osiąga lepszych wyników”, tylko mniej kosztuje.

**2. Jaki koszt był najbardziej odczuwalny?**

Pamięć, i to względnie. Plik `gemma3:4b` jest 4,2× większy niż `gemma3:1b` (3,12 wobec 0,75 GB), a po załadowaniu 2,6× (2,09 wobec 0,81 GB). Dla kwantyzacji Q8_0 względem Q4_K_M pamięć rośnie 2× (4,13 wobec 2,09 GB). Pozostałe koszty są mniejsze: szybkość generowania spada ok. 2× przy przejściu z 1b na 4b (ok. 207 wobec 105 tokenów/s) i o ok. 10% przy przejściu z Q4_K_M na Q8_0, a TTFT w stanie ciepłym to różnica rzędu dziesięciu milisekund (np. 17 wobec 29 ms), niezauważalna dla użytkownika. W wartościach bezwzględnych najbardziej widoczny jest cold start (od 0,3 do 2,3 s), ale dotyczy tylko pierwszego zapytania.

**3. Czy kwantyzacja zmieniła jakość na tyle, by miało to znaczenie praktyczne?**

Statystycznie tak, praktycznie niewiele. Q8_0 podniosło macro-F1 z 0,456 do 0,494 (+0,038), a sparowany 95% przedział różnicy [+0,010; +0,070] nie zawiera zera. Q4_K_M i Q8_0 dają te same predykcje dla 92,25% dokumentów. Z 31 dokumentów, na których się różnią, 15 zmieniło się z błędnych na poprawne, a 7 z poprawnych na błędne, czyli netto 8 dokumentów (+2 punkty procentowe accuracy). Za to koszt jest wyraźny: 2× więcej pamięci i o ok. 10% wolniejsze generowanie. Przy tak niskim poziomie wyjściowym (macro-F1 ok. 0,5) taka poprawa nie zmienia oceny przydatności modelu.

**4. Czy odpowiedzi poprawne formalnie były też poprawne merytorycznie?**

Nie. Na 6 przypadkach strukturalnych (dla każdego z dwóch modeli):

| tryb | rekordy z poprawnym JSON | z tego z błędną treścią (1b / 4b) |
|---|---:|---:|
| `free` | 0 z 12 (oba modele zawsze owijają JSON w znaczniki markdown) | po usunięciu znaczników: 5 z 6 / 3 z 6 błędnych |
| `json` | 12 z 12 | 6 z 6 / 3 z 6 błędnych |
| `schema` | 12 z 12 | 6 z 6 / 3 z 6 błędnych |

Wymuszenie formatu (`json`, `schema`) daje 100% poprawnej składni i zgodności ze schematem, ale nie poprawia treści: błędy to głównie zamiana formy odmienionej na mianownik („Poznań” zamiast „Poznaniu”), stanowisko wzięte za osobę („Prezes Orlenu”) i dla `gemma3:1b` wartości-śmieci (napis „null”, „nieznany”). W trybie `free` błędem jest format, nie treść. Dla `gemma3:1b` wymuszenie formatu nawet pogorszyło treść (poprawność pól po usunięciu znaczników w trybie `free` to 0,56, a w `json` i `schema` 0,28).

**5. Jakie dwa ograniczenia benchmarku najbardziej utrudniają uogólnienie?**

1. **Mała i wąska próbka.** Zadanie strukturalne ma tylko 6 przypadków (jeden przypadek to 16,7 punktu procentowego), a klasyfikacja 400 dokumentów z dwóch domen (recenzje lekarzy i hoteli, teksty prasowe), po jednym uruchomieniu.
2. **Zero-shot z jednym, zamrożonym promptem.** Modele nie widziały żadnego przykładu ani definicji klas, więc nie mogą wiedzieć, co zbiór rozumie przez `niejednoznaczna` (stąd prawie zerowe F1 tej klasy). Wynik zależy od brzmienia promptu, którego z założenia nie dostrajaliśmy osobno dla każdego modelu. Porównanie z baseline'em jest przez to nierówne: baseline uczył się na 5782 przykładach, LLM na żadnym.

# Zadanie 10: Decyzja projektowa

To zadanie nie wymaga nowej implementacji.

Proszę wybrać model dla dwóch scenariuszy:

1. lokalny system, w którym najważniejsza jest jakość;
2. lokalny system, w którym ważne są mała pamięć i szybka odpowiedź.

Dla każdego wyboru proszę podać wynik jakościowy, koszt, najważniejszy typ błędu i jedno ograniczenie.

Na końcu proszę odpowiedzieć: **kiedy lokalny LLM jest uzasadnionym wyborem, a kiedy prostszy model zadaniowy pozostaje lepszym rozwiązaniem?**


### Odpowiedzi do zadania 10

**1. System, w którym najważniejsza jest jakość: `gemma3:4b-it-q8_0` (Q8_0).**

- **Wynik:** macro-F1 = 0,494 (95% CI [0,449; 0,539]), najlepszy spośród testowanych LLM-ów (Q4_K_M: 0,456, `gemma3:1b`: 0,349). Poprawność JSON: 1,0, poprawność całego rekordu 0,5.
- **Koszt:** 4,65 GB na dysku, 4,13 GB pamięci po załadowaniu, ok. 224 ms na dokument, generowanie o ok. 10% wolniejsze niż w Q4_K_M.
- **Najważniejszy błąd:** zła etykieta. Modele wybierają dominujący ton i prawie nigdy nie przewidują klasy `niejednoznaczna`, a teksty informacyjne (`neutralna`) często uznają za pozytywne (dla `gemma3:4b` Q4_K_M: 1 z 100 poprawnych w `niejednoznaczna`, 29 z 100 w `neutralna`; Q8_0 daje te same predykcje w 92% przypadków). W zadaniu strukturalnym model zamienia formy odmienione na mianownik.
- **Ograniczenie:** przewaga nad Q4_K_M jest mała (+0,038 macro-F1, sparowany CI [+0,010; +0,070]), a kosztuje 2× więcej pamięci. Cały wynik jest wciąż wyraźnie niższy niż klasyczny baseline (0,793).

**2. System, w którym ważne są mała pamięć i szybka odpowiedź: `gemma3:1b`.**

- **Wynik:** macro-F1 = 0,349 (95% CI [0,318; 0,382]), accuracy 0,475. Poprawność całego rekordu w zadaniu strukturalnym: 0 z 6.
- **Koszt:** 0,75 GB na dysku i 0,81 GB w pamięci (ok. 2,6× mniej niż `gemma3:4b`), ok. 207 tokenów/s i TTFT w stanie ciepłym ok. 17 ms (ok. 2× szybciej niż `gemma3:4b`).
- **Najważniejszy błąd:** praktycznie nie rozpoznaje klas `neutralna` (7 z 100 poprawnych) i `niejednoznaczna` (0 z 100). W zadaniu strukturalnym dopisuje wartości-śmieci („null” jako napis, „nieznany”) i rozbija nazwy własne.
- **Ograniczenie:** jakość jest zbyt niska do zastosowań, w których pomyłki mają znaczenie. Szybkość i mała pamięć nie rekompensują macro-F1 poniżej 0,4 przy czterech klasach.

**Kiedy lokalny LLM jest uzasadnionym wyborem, a kiedy prostszy model zadaniowy jest lepszy?**

Lokalny LLM jest uzasadniony, gdy dane wejściowe są bardzo zróżnicowane (różne dziedziny, formaty, style), gdy nie ma oznaczonych danych treningowych albo zadanie lub schemat wyniku często się zmieniają, bo wystarczy wtedy zmiana promptu zamiast nowego trenowania. Płacimy za to jakością, pamięcią i czasem. Prostszy model zadaniowy jest lepszy, gdy zadanie jest stałe, są dane treningowe, a dane wejściowe są podobne do treningowych: w naszym eksperymencie TF-IDF z regresją logistyczną uzyskał macro-F1 0,793 wobec 0,494 dla najlepszego LLM-a, działa ok. 3500× szybciej (0,064 wobec 224 ms na dokument) i zajmuje 3,46 MB na dysku zamiast 4,65 GB. Trzeba jednak pamiętać, że to porównanie nie jest symetryczne (baseline uczył się na 5782 przykładach, LLM działa zero-shot), a odporność obu podejść na zmianę dziedziny przewidujemy, ale w tym laboratorium jej nie mierzyliśmy.

# Karta modelu

Proszę utworzyć `artifacts/local_llm_model_card.md` dla modelu rekomendowanego do dalszych laboratoriów.

Karta powinna zawierać: nazwę i kwantyzację, środowisko, pamięć i rozmiar, TTFT i tokeny/s, wyniki klasyfikacji, wyniki wyjścia strukturalnego, porównanie z baseline'em, dominujące błędy z analizy 20 przypadków, zastosowanie, ograniczenia oraz uzasadnienie wyboru.


# Wnioski

Proszę zapisać KILKA krótkich wniosków opartych na uzyskanych liczbach i przykładach. Wnioski powinny obejmować jakość, koszt, kwantyzację, wyjście strukturalne oraz relację LLM–baseline.


### Wnioski

1. **Jakość.** Większy model jest lepszy, ale nie we wszystkim: `gemma3:4b` ma macro-F1 0,456 wobec 0,349 dla `gemma3:1b`, a najlepszy jest `gemma3:4b-it-q8_0` (0,494). Wszystkie modele prawie nigdy nie rozpoznają klasy `niejednoznaczna` i często uznają teksty informacyjne za pozytywne, więc rozmiar tego błędu nie usuwa.
2. **Koszt.** Najbardziej odczuwalna jest pamięć: `gemma3:4b` zajmuje 4,2× więcej na dysku i 2,6× więcej po załadowaniu niż `gemma3:1b`, a Q8_0 ma 2× większe zużycie pamięci niż Q4_K_M. Generowanie jest ok. 2× wolniejsze przy przejściu z 1b na 4b, a różnice TTFT w stanie ciepłym to dziesiątki milisekund.
3. **Kwantyzacja.** Q8_0 podniosło macro-F1 z 0,456 do 0,494 (+0,038, sparowany CI [+0,010; +0,070]), więc różnica jest statystycznie istotna, ale praktycznie niewielka: obie wersje dają te same predykcje w 92,25% przypadków, a netto poprawiły się tylko 8 dokumentów na 400. Kosztem jest 2× więcej pamięci.
4. **Wyjście strukturalne.** Wymuszenie formatu (`json`, `schema`) daje 100% poprawnej składni, ale nie poprawia treści: tylko 3 z 6 rekordów jest poprawnych dla `gemma3:4b` i 0 z 6 dla `gemma3:1b`. W trybie `free` oba modele owijają JSON w znaczniki markdown (poprawność składni 0,0). Typowe błędy treści to sprowadzanie nazw do mianownika i branie stanowiska za osobę.
5. **LLM a baseline.** Klasyczny TF-IDF z regresją logistyczną ma macro-F1 0,793 wobec 0,494 dla najlepszego LLM-a, jest ok. 3500× szybszy i zajmuje ok. 80× mniej pamięci. Porównanie nie jest symetryczne (baseline uczył się na 5782 przykładach, LLM działa zero-shot), więc LLM ma sens tam, gdzie brakuje danych treningowych albo zadanie często się zmienia.


# *Zadanie dla chętnych: polska fleksja i znaki diakrytyczne

Zadanie nie jest wymagane do zaliczenia. Sprawdza dwie rzeczy, których nie widać na przykład w benchmarkach anglojęzycznych.

## Część A: forma podstawowa encji

W zadaniu 5 model kopiował fragment tekstu. Teraz ma zwrócić encję w mianowniku: `Krakowie` → `Kraków`, `Polskiego Związku Narciarskiego` → `Polski Związek Narciarski`. Wymaga to wygenerowania poprawnej formy, a nie przepisania.

Proszę uruchomić `LEMMA_CASES` z promptem `LEMMA_PROMPT` w trybie `format=IE_SCHEMA` dla co najmniej dwóch modeli i policzyć:

- poprawność pól i całego rekordu (`score_ie` względem `gold_lemma`);
- rozkład typów błędów z `lemma_error_type` dla każdego modelu;
- poprawność osobno dla nazw jednowyrazowych i wielowyrazowych.

Należy uruchomić i przeprowadzić dwa osobne eksperymenty na tych samych ośmiu tekstach:
- Ekstrakcja literalna z IE_PROMPT, oceniana względem gold.
- Normalizacja do mianownika z LEMMA_PROMPT, oceniana względem gold_lemma.

Proszę odpowiedzieć:

1. O ile spada poprawność pól względem wariantu literalnego (ten sam przypadek oceniony względem `gold`)?
2. Który typ błędu dominuje: przepisanie formy odmienionej czy wygenerowanie formy błędnej?
3. Które zjawiska są najtrudniejsze: nazwy wielowyrazowe z uzgodnieniem, nazwy w liczbie mnogiej (`Kielc`), odmiana przymiotnikowa (`Zakopanego`), nazwa osoby wewnątrz nazwy organizacji?
4. Czy większy model lub wyższa kwantyzacja zmienia wynik bardziej niż w zadaniu 5?

## Część B: tekst bez znaków diakrytycznych

Użytkownicy często piszą bez polskich znaków. Proszę przepuścić `classification_sample` przez `strip_diacritics` i powtórzyć klasyfikację z zadania 4 dla jednego LLM-a oraz dla baseline'u `TF-IDF + Linear SVM` (bez ponownego trenowania).

Dla każdego systemu proszę podać:

- macro-F1 na tekście oryginalnym i bez diakrytyków;
- 95% przedział ufności sparowanej różnicy (`paired_macro_f1_diff_ci`);
- liczbę zmian „poprawna → błędna" i „błędna → poprawna";
- dla LLM-a: średnią `prompt_tokens` w obu wariantach.

Proszę odpowiedzieć:

1. Który system traci więcej i dlaczego? Proszę odnieść się do słownika TF-IDF i do tokenizacji podwyrazowej.
2. Czy usunięcie diakrytyków zmienia liczbę tokenów promptu, a więc koszt?
3. Proszę pokazać trzy dokumenty, w których predykcja się zmieniła, i wskazać słowa, które mogły o tym zdecydować (np. `łaska` / `laska`, `sąd` / `sad`).


In [ ]:
# Część A: gold = fragment literalny, gold_lemma = forma podstawowa (mianownik).
LEMMA_CASES = [
    {"text": "Marii Skłodowskiej-Curie poświęcono wystawę w Paryżu.",
     "gold": {"person": "Marii Skłodowskiej-Curie", "organization": None, "location": "Paryżu"},
     "gold_lemma": {"person": "Maria Skłodowska-Curie", "organization": None, "location": "Paryż"}},
    {"text": "Do Gdańska przyjechał Adam Małysz na zaproszenie Polskiego Związku Narciarskiego.",
     "gold": {"person": "Adam Małysz", "organization": "Polskiego Związku Narciarskiego", "location": "Gdańska"},
     "gold_lemma": {"person": "Adam Małysz", "organization": "Polski Związek Narciarski", "location": "Gdańsk"}},
    {"text": "Prezes Orlenu spotkał się z dziennikarzami w Płocku.",
     "gold": {"person": None, "organization": "Orlenu", "location": "Płocku"},
     "gold_lemma": {"person": None, "organization": "Orlen", "location": "Płock"}},
    {"text": "Nagrodę wręczono Januszowi Gajosowi w Teatrze Narodowym w Warszawie.",
     "gold": {"person": "Januszowi Gajosowi", "organization": "Teatrze Narodowym", "location": "Warszawie"},
     "gold_lemma": {"person": "Janusz Gajos", "organization": "Teatr Narodowy", "location": "Warszawa"}},
    {"text": "W Zielonej Górze spotkali się przedstawiciele Polskich Kolei Państwowych.",
     "gold": {"person": None, "organization": "Polskich Kolei Państwowych", "location": "Zielonej Górze"},
     "gold_lemma": {"person": None, "organization": "Polskie Koleje Państwowe", "location": "Zielona Góra"}},
    {"text": "Kibice z Kielc czekali na Roberta Lewandowskiego.",
     "gold": {"person": "Roberta Lewandowskiego", "organization": None, "location": "Kielc"},
     "gold_lemma": {"person": "Robert Lewandowski", "organization": None, "location": "Kielce"}},
    {"text": "Wycieczka do Zakopanego kosztowała Annę Kowalską majątek.",
     "gold": {"person": "Annę Kowalską", "organization": None, "location": "Zakopanego"},
     "gold_lemma": {"person": "Anna Kowalska", "organization": None, "location": "Zakopane"}},
    {"text": "Delegacja Uniwersytetu Marii Curie-Skłodowskiej odwiedziła Katowice.",
     "gold": {"person": None, "organization": "Uniwersytetu Marii Curie-Skłodowskiej", "location": "Katowice"},
     "gold_lemma": {"person": None, "organization": "Uniwersytet Marii Curie-Skłodowskiej", "location": "Katowice"}},
]

LEMMA_PROMPT = '''Wyodrębnij z polskiego tekstu jedną osobę, jedną organizację i jedną lokalizację.
Każdą nazwę podaj w formie podstawowej (mianownik), a nie w formie użytej w tekście. Jeśli kategorii nie ma, zwróć null.
Odpowiedz wyłącznie obiektem JSON z kluczami 'person', 'organization', 'location'.

Tekst:
'''


def lemma_error_type(pred, literal, lemma):
    """Typ wyniku dla jednego pola w części A."""
    if pred == lemma:
        return "OK"
    if lemma is None:
        return "HALLUCINATION"        # model podał encję, której nie ma
    if pred is None:
        return "MISSING"
    if pred == literal:
        return "COPIED_INFLECTED"     # przepisana forma odmieniona
    return "WRONG_FORM"               # inna, błędna forma lub inny fragment


# Część B: ł nie rozkłada się w Unicode, dlatego używamy jawnej tabeli.
_PL_DIACRITICS = str.maketrans("ąćęłńóśźżĄĆĘŁŃÓŚŹŻ", "acelnoszzACELNOSZZ")


def strip_diacritics(text):
    return str(text).translate(_PL_DIACRITICS)


assert strip_diacritics("Zażółć gęślą jaźń") == "Zazolc gesla jazn"
assert lemma_error_type("Krakowie", "Krakowie", "Kraków") == "COPIED_INFLECTED"
assert lemma_error_type("Kraków", "Krakowie", "Kraków") == "OK"

# Warunki zaliczenia laboratorium

- [ ] audyt środowiska i co najmniej dwóch modeli;
- [ ] oszacowanie i pomiar pamięci;
- [ ] cold start, TTFT i tokeny/s;
- [ ] benchmark klasyfikacji zero-shot;
- [ ] benchmark wyjścia strukturalnego w trzech trybach;
- [ ] surowe odpowiedzi zapisane w `../artifacts`;
- [ ] przedział ufności macro-F1 i porównanie sparowane kwantyzacji;
- [ ] porównanie dwóch kwantyzacji lub dwóch rozmiarów modelu;
- [ ] porównanie z `TF-IDF + Linear SVM`;
- [ ] tabela dokładnie 20 przypadków;
- [ ] odpowiedzi do dwóch zadań interpretacyjnych;
- [ ] `artifacts/local_llm_model_card.md`;
- [ ] wnioski.

Dla chętnych: zadanie o polskiej fleksji i znakach diakrytycznych (część A, część B lub obie).